**Tokenizer**

- byte pair encoding (BPE)
  - encode the sequence
  - get the most likely pairs of bytes, then merge them (e.g. (101,104) -> 257)
  - add that pair to a list called 'merges' and add it to your vocab dict
  - do that num_merges times
- when you want to encode something
  - encode it normally in utf-8
  - then walk the 'merges' list using a for loop - and begin merging tokens on every pass until you've done all the 'merges' in the 'merges' list
  - its important you walk the list in a for loop, because some merges may be of (257, 102) for example, which is a merging of an already merged token (i.e. 257)

_Here's an intuitive diagram!_

- each step is a pass of the text data, where we merge the first pair in our 'merges' list, then the second...
- we do this $\text{num merges}$ times, since that's how many merges we did during training to construct our new vocab_size!

_PS: GPT-2's vocab_size of 50,257 = 50,000 merges + 256 base bytes + **1** special token_

- that one token is `<|endoftext|>`, and it does three jobs at once: document separator, BOS, and EOS. they're all the same signal — "a document boundary is here" — so GPT-2 needs no separate BOS token
- there is **no** padding token. pretraining concatenates the whole corpus into one stream and slices fixed-length windows, so every window is the same length by construction and nothing is ever ragged. padding is only needed for variable-length data (fine-tuning, batched inference)
- special tokens can't be _learned_ by BPE — every merged token is by definition real text that appeared in the training data. if `<|endoftext|>` were reachable that way, a user could type it and forge a document boundary. so it's registered separately, at an id above every merge

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQUQ8_fwiCtMQr-aVf-Ii1mhRPCccnz6627SLUhUmwtYJFMSWmAAoALVz0Q&s=10" width="400">


**What to consider when tokenizing**

Vocab size vs encoding efficiency

- you can have a few ids (i.e. 256 for example using default Unicode 8 encoding), which is one byte for every character
- BUT that means you'll need a lot of token_ids to represent one string of text (i.e. 10,000 character extract -> 10,000 token_ids)

The trade is to have more token_ids, and more encoding efficiency

- now we could expand our vocabulary (i.e. from 256 -> 50,257 word encodings) meaning we'd go from one byte per character to 1-2 bytes representing 3-4+ characters (depending on the relative frequency of characters and pairs in our training data)
- now it might take only 4,000 token_ids to encode a 10,000 character extract

_Why this is good?_

- computationally cheaper - your sequence length is smaller so the next token you predict is a bigger chunk of text
- much easier for the model to learn relationships - consider trying to embed the semantic meaning of '-able' vs '-b', like what even is the semantic meaning of the letter 'b'???

_Why is this bad_

- our LM head and our embedding matrix are ENORMOUS! That's why early GPT-2 used weight-tying (i.e. make them the exact same weights)


In [ ]:
with open('data/input.txt', 'r') as f:
	data = f.read()

In [10]:
import re
from collections import defaultdict


class BPETokenizer:

	def __init__(self, vocab_size:int):

		self.merge_mapping = {}
		assert vocab_size >= 256, f"BPE needs vocab size greater than 256"

		self.vocab = {i: bytes([i]) for i in range(256)} # mapping from byte to index - initially its just 1:1, 2:2, but after merging you will get 259: (223, 24)
		self.merges = defaultdict(int)

		# special tokens live ABOVE the merge range so a merge can never collide with one.
		# BPE only builds tokens by concatenating existing tokens (bottoming out in the 256 raw
		# bytes), so every BPE token is real text that appeared in training. a special token must
		# NOT be reachable that way - else typing "<|endoftext|>" would forge a document boundary
		self.special_tokens = {} # str -> int
		self.inverse_special = {} # int -> str

	@property
	def vocab_size(self):
		return len(self.vocab) # 256 bytes + merges + specials

	def get_stats(self, ids):

		# then of those tokens, now merge the ones that have mergeed pairs
		counts = defaultdict(int)
		for pair in zip(ids, ids[1:]):
			counts[pair] += 1

		return counts

	def merge_seq(self, ids, pair, idx):
		newids = []
		i = 0
		while i < len(ids):
			if i < len(ids) - 1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
				i += 2
				newids.append(idx)
			else:
				newids.append(ids[i]) # append the current token
				i += 1 # move to the next
		return newids
	
	def merge(self, ids, num_merges: int):

		# TRAINING - call this once. encode() must NOT call it: encoding should only apply merges
		# already learned, otherwise the vocab changes with whatever text you happened to encode
		
		for i in range(num_merges):
			stats = self.get_stats(ids) # was self.counts, which never existed
			if not stats: # sequence collapsed to <2 tokens, nothing left to merge
				break
			freq_pair = max(stats, key=stats.get) # most frequent pair
			idx = 256 + i
			ids = self.merge_seq(ids, freq_pair, idx)
			self.merges[freq_pair] = idx
		
		for (p0, p1), idx in self.merges.items():
			
			# concat bytes objects - vocab[p0] is ALREADY bytes (seeded by bytes([i]) above),
			# so this is a lookup + concat, not a conversion
			self.vocab[idx] = self.vocab[p0] + self.vocab[p1]

		return ids

	def register_special_tokens(self, tokens):

		# call AFTER merge(), so these ids sit above every merged token
		for tok in tokens:
			idx = len(self.vocab)
			self.special_tokens[tok] = idx
			self.inverse_special[idx] = tok
			self.vocab[idx] = tok.encode('utf-8') # mirror into vocab, else decode() KeyErrors on it

	def decode(self, ids):
		
		# decode a sequence 
		tokens = b"".join(self.vocab[int(idx)] for idx in ids)
		text = tokens.decode("utf-8", errors="replace") # replace: a batch slice can cut a multi-byte char in half
		return text

	def _encode(self, text):

		ids = text.encode('utf-8') # returns array of bytes

		for pair, idx in self.merges.items():
			ids = self.merge_seq(ids, pair, idx)

		return ids

	def encode(self, text, allowed_special=True):

		# allowed_special=False for untrusted text - then a literal "<|endoftext|>" tokenizes as
		# ordinary bytes and cannot forge a document boundary
		if not allowed_special or not self.special_tokens:
			return self._encode(text)

		# capturing group keeps the delimiters in the split output; re.escape because
		# "<|endoftext|>" contains |, which triggers regex specific syntax
		pattern = "(" + "|".join(re.escape(t) for t in self.special_tokens) + ")"

		ids = []
		for chunk in re.split(pattern, text): 
			if chunk in self.special_tokens:
				ids.append(self.special_tokens[chunk])
			elif chunk:
				ids.extend(self._encode(chunk))

		return ids

bpe = BPETokenizer(vocab_size=256)
bpe.merge(list(data.encode('utf-8')), num_merges=50) # train first
bpe.register_special_tokens(["<|endoftext|>"])

with open(os.path.join('data/tokenizer.json'), 'w') as f: 
	json.dump({
		'merges' : [[list(p), i] for p, i in bpe.merges.items()], # must be saved as list since tuples aren't able to be put into json
		'special_tokens': bpe.special_tokens,
	}, f)

# in order to load self.merges, self.special..., self.inverse... when trying to reinitialize the tokenizer
with open(os.path.join('data/tokenizer.json')) as f:
	tok_json = json.load(f)
	bpe.merges = {tuple(p): i for p, i in tok_json['merges']} # converts list into a tuple, then reconstructs the merges blob
	bpe.special_tokens = tok_json['special_tokens']
	bpe.inverse_special = tok_json['inverse_special']

encoded = bpe.encode(data)
assert bpe.decode(encoded) == data, "round trip failed"
print(f"vocab_size={bpe.vocab_size}, {len(data)} chars -> {len(encoded)} tokens")
bpe.decode(encoded)[:80]

vocab_size=307, 1115394 chars -> 773537 tokens


'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.'